In [0]:
%pip install openpyxl

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# 1. Define the direct download URL for the SINESP Excel file
url_sinesp = "https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/download/dnsp-base-de-dados/bancovde-2026.xlsx/@@download/file"
temp_file_path = "/tmp/sinesp_data_2026.xlsx"

print("Starting SINESP Excel file download...")

# 2. Download the file and save it temporarily in the cluster
response = requests.get(url_sinesp, verify=False)

if response.status_code == 200:
    with open(temp_file_path, "wb") as file:
        file.write(response.content)
    print("Download successful. Reading Excel file with Pandas...")
    
    # 3. Read the Excel file using Pandas
    # 'openpyxl' is the standard engine for .xlsx files (now installed via %pip)
    pdf_sinesp = pd.read_excel(temp_file_path, engine='openpyxl')
    
    # 4. Convert Pandas DataFrame to PySpark DataFrame
    df_raw_sinesp = spark.createDataFrame(pdf_sinesp.astype(str))
    
    # 5. Add Audit Metadata Columns
    df_bronze_sinesp = df_raw_sinesp.withColumn("ingestion_timestamp", current_timestamp()) \
                                    .withColumn("source_system", lit("SINESP_GOV_BR_XLSX"))
                                    
    # 6. Save as a Managed Delta Table (Bronze Layer)
    table_name_bronze_sinesp = "bronze_sinesp_crime_data"

    df_bronze_sinesp.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name_bronze_sinesp)

    print(f"Bronze load complete! {df_bronze_sinesp.count()} records saved to managed table '{table_name_bronze_sinesp}'.")
    display(df_bronze_sinesp)

else:
    print(f"Failed to download the file. HTTP Status Code: {response.status_code}")